In [1]:
from pyspark.sql import SparkSession
import pyspark

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("Lakehouse-Iceberg-TrainModel")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")


In [2]:
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")


In [3]:
query = """
SELECT 
    f.time_id,
    f.customer_id,
    f.product_id,
    f.location_id,
    f.quantity,
    f.unit_price,        
    f.total_revenue,     

    -- Dim_time
    t.order_date,
    t.year,
    t.month,
    t.day,
    t.quarter,
    t.weekday_name,

    -- Dim_customer
    c.gender,
    c.age_group,
    c.education,
    c.income,
    c.race,
    c.state,             -- Cột state từ bảng khách hàng

    -- Dim_product
    p.product_title,
    p.product_category,

    -- Dim_location
    l.state_name purchase_state,
    l.region purchase_region

FROM nessie.gold_db.fact_order f
LEFT JOIN nessie.gold_db.dim_time t     ON f.time_id     = t.time_id
LEFT JOIN nessie.gold_db.dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN nessie.gold_db.dim_product p   ON f.product_id  = p.product_id
LEFT JOIN nessie.gold_db.dim_location l  ON f.location_id = l.location_id
"""

In [4]:
df_fact_full = spark.sql(query)
df_fact_full.limit(10).toPandas()


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,time_id,customer_id,product_id,location_id,quantity,unit_price,total_revenue,order_date,year,month,...,gender,age_group,education,income,race,state,product_title,product_category,purchase_state,purchase_region
0,20200420,R_3O9aPnM7ixjsbYK,1250153239,1,1.0,14.04,14.04,2020-04-20,2020,4,...,Female,35 - 44 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Idaho,"Real Life Dinners: Fun, Fresh, Fast Dinners fr...",ABIS_BOOK,Idaho,West
1,20210103,R_1n6GpJR7EUFnaWr,1523509651,1,1.0,9.89,9.89,2021-01-03,2021,1,...,Male,35 - 44 years,Bachelor's degree,"$50,000 - $74,999",White or Caucasian,Ohio,Paint by Sticker: Dogs: Create 12 Stunning Ima...,ABIS_BOOK,Idaho,West
2,20211220,R_22Psw7KTkXTMEGl,0679408525,1,1.0,16.79,16.79,2021-12-20,2021,12,...,Female,35 - 44 years,High school diploma or GED,"$25,000 - $49,999",White or Caucasian,Idaho,Field Guide to the Night Sky (National Audubon...,ABIS_BOOK,Idaho,West
3,20210617,R_2frj956dhgNpdqD,1421569078,1,1.0,17.99,17.99,2021-06-17,2021,6,...,Male,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Idaho,"Monster: The Perfect Edition, Vol. 2 (2)",ABIS_BOOK,Idaho,West
4,20200815,R_2frj956dhgNpdqD,1506715214,1,1.0,33.84,33.84,2020-08-15,2020,8,...,Male,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Idaho,Berserk Deluxe Volume 4,ABIS_BOOK,Idaho,West
5,20190210,R_8cr0J12EMKs9NND,1793495564,1,1.0,9.99,9.99,2019-02-10,2019,2,...,Female,55 - 64 years,High school diploma or GED,"Less than $25,000",White or Caucasian,Missouri,A Gift of Gratitude: A Community Book Project,ABIS_BOOK,Idaho,West
6,20180328,R_8cr0J12EMKs9NND,1493028499,1,1.0,3.73,3.73,2018-03-28,2018,3,...,Female,55 - 64 years,High school diploma or GED,"Less than $25,000",White or Caucasian,Missouri,The Golden Elixir of the West: Whiskey and the...,ABIS_BOOK,Idaho,West
7,20200711,R_2frj956dhgNpdqD,178909156X,1,1.0,25.99,25.99,2020-07-11,2020,7,...,Male,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Idaho,The Art of Death Stranding,ABIS_BOOK,Idaho,West
8,20200809,R_2frj956dhgNpdqD,1506712002,1,1.0,37.62,37.62,2020-08-09,2020,8,...,Male,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Idaho,Berserk Deluxe Volume 3,ABIS_BOOK,Idaho,West
9,20200627,R_3O9aPnM7ixjsbYK,052557588X,1,1.0,13.55,13.55,2020-06-27,2020,6,...,Female,35 - 44 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Idaho,"Tasty Every Day: All of the Flavor, None of th...",ABIS_BOOK,Idaho,West


In [5]:
# Hiển thị schema sau khi làm sạch
df_fact_full.printSchema()

root
 |-- time_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- location_id: long (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- weekday_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- education: string (nullable = true)
 |-- income: string (nullable = true)
 |-- race: string (nullable = true)
 |-- state: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- purchase_state: string (nullable = true)
 |-- purchase_region: string (nullable = true)



In [6]:
# Đếm số dòng
num_rows =df_fact_full.count()
# Đếm số cột
num_cols = len(df_fact_full.columns)
print(f"\nKích thước dữ liệu: ({num_rows}, {num_cols})")


Kích thước dữ liệu: (1675015, 23)


In [7]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator

# =====================================================
# Tính toán toàn bộ feature dựa trên Schema Gold chuẩn
# =====================================================
df_final = (
    df_fact_full.groupBy("customer_id")
    .agg(
        # Tổng chi tiêu (Đã đổi sang total_revenue)
        F.sum("total_revenue").alias("total_spend"),

        # Trung bình chi tiêu trên mỗi đơn hàng
        (F.sum("total_revenue") / F.countDistinct("time_id")).alias("avg_order_value"),

        # Độ lệch chuẩn chi tiêu trên đơn hàng
        F.stddev("total_revenue").alias("std_order_value"),

        # Trung bình số lượng sản phẩm mỗi đơn
        (F.sum("quantity") / F.countDistinct("time_id")).alias("avg_items_per_order"),

        # Tổng số đơn hàng
        F.countDistinct("time_id").alias("total_orders"),

        # Năm đầu và năm cuối hoạt động
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year")
    )
    # Thêm số năm hoạt động và chi tiêu trung bình mỗi năm
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    .withColumn("avg_spend_per_year", F.col("total_spend") / F.col("years_active"))
    .na.fill(0)  # Đảm bảo không có null để tránh lỗi khi đưa vào mô hình ML
)

# Kiểm tra dữ liệu sau khi tính toán feature
df_final.show(5)

+-----------------+------------------+------------------+------------------+-------------------+------------+----------+---------+------------+------------------+
|      customer_id|       total_spend|   avg_order_value|   std_order_value|avg_items_per_order|total_orders|first_year|last_year|years_active|avg_spend_per_year|
+-----------------+------------------+------------------+------------------+-------------------+------------+----------+---------+------------+------------------+
|R_1jO4s7oht3pyKEc|13041.710000000001| 60.37828703703704|25.637963404064184| 2.8333333333333335|         216|      2018|     2023|           6|2173.6183333333333|
|R_10TV1zyi4yCEEkl|30578.989999999994| 74.40143552311434| 83.67537499523931|   2.67639902676399|         411|      2018|     2023|           6| 5096.498333333332|
|R_2dyITPHbbfmCXJn|16020.080000000004| 92.06942528735634| 49.11779146550432| 3.1666666666666665|         174|      2018|     2023|           6| 2670.013333333334|
|R_3GD1CL4OyjglmbZ|   

In [8]:

df_final.select(
    "customer_id",
    "total_spend", "avg_spend_per_year",
    "avg_order_value", "std_order_value",
    "avg_items_per_order", "total_orders",
    "years_active"
).show(10, truncate=False)


+-----------------+------------------+------------------+------------------+------------------+-------------------+------------+------------+
|customer_id      |total_spend       |avg_spend_per_year|avg_order_value   |std_order_value   |avg_items_per_order|total_orders|years_active|
+-----------------+------------------+------------------+------------------+------------------+-------------------+------------+------------+
|R_1jO4s7oht3pyKEc|13041.710000000001|2173.6183333333333|60.37828703703704 |25.637963404064184|2.8333333333333335 |216         |6           |
|R_10TV1zyi4yCEEkl|30578.989999999994|5096.498333333332 |74.40143552311434 |83.67537499523931 |2.67639902676399   |411         |6           |
|R_2dyITPHbbfmCXJn|16020.080000000004|2670.013333333334 |92.06942528735634 |49.11779146550432 |3.1666666666666665 |174         |6           |
|R_3GD1CL4OyjglmbZ|17528.47          |2921.411666666667 |39.3898202247191  |34.54485118060136 |1.750561797752809  |445         |6           |
|R_297

In [9]:
# =====================================================
#  Xác định features & target
# =====================================================
num_features = [
    "total_spend",
    "years_active",
    "avg_order_value",
    "std_order_value",
    "avg_items_per_order",
    "total_orders"
]
target = "avg_spend_per_year"

df_clean = df_final.na.drop(subset=num_features + [target])
# Chia train/test
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()} dòng | Test: {test_df.count()} dòng")

assembler = VectorAssembler(
    inputCols=num_features,
    outputCol="features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

Train: 3988 dòng | Test: 972 dòng


## Random Forest

In [10]:
# --------------------------
# Pipeline mô hình Random Forest
# --------------------------
rf = RandomForestRegressor(
    labelCol=target,
    featuresCol="scaledFeatures",
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

# --------------------------
# Tập siêu tham số cần thử
# --------------------------
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 200])
    .addGrid(rf.maxDepth, [8, 10, 12])
    .build()
)

# --------------------------
# TrainValidationSplit
# --------------------------
evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")

tvs = TrainValidationSplit(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=4
)

# --------------------------
# Huấn luyện mô hình
# --------------------------
rf_tvs_model = tvs.fit(train_df)

# --------------------------
# In thông tin mô hình tốt nhất
# --------------------------
best_rf = rf_tvs_model.bestModel.stages[-1]
print("Best Model:")
print(" - numTrees:", best_rf.getNumTrees)
print(" - maxDepth:", best_rf.getMaxDepth())


Best Model:
 - numTrees: 50
 - maxDepth: 10


In [11]:
# --------------------------
# Đánh giá mô hình
# --------------------------
train_pred = rf_tvs_model.bestModel.transform(train_df)
test_pred  = rf_tvs_model.bestModel.transform(test_df)

metrics = ['r2', 'mae', 'rmse']
for metric in metrics:
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    print(f"{metric.upper()} Train:", round(evaluator.evaluate(train_pred), 4))
    print(f"{metric.upper()} Test :", round(evaluator.evaluate(test_pred), 4))
    print("----------")

R2 Train: 0.9881
R2 Test : 0.9792
----------
MAE Train: 74.1848
MAE Test : 103.3177
----------
RMSE Train: 181.9184
RMSE Test : 228.1795
----------


In [12]:
import mlflow
import mlflow.spark
from pyspark.ml.evaluation import RegressionEvaluator

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Spending_Prediction_V1")

# Tính metrics train/test
metrics_dict = {}
for metric in metrics:  # metrics = ['r2', 'mae', 'rmse']
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_dict[f"{metric}_train"] = round(evaluator.evaluate(train_pred), 4)
    metrics_dict[f"{metric}_test"]  = round(evaluator.evaluate(test_pred), 4)

# Lưu vào MLflow
with mlflow.start_run(run_name="RandomForest_Model"):
    # Log param
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("numTrees", best_rf.getNumTrees)
    mlflow.log_param("maxDepth", best_rf.getMaxDepth())
    
    # Log metric
    for k, v in metrics_dict.items():
        mlflow.log_metric(k, v)
    
    # Log mô hình (PipelineModel đầy đủ)
    mlflow.spark.log_model(rf_tvs_model.bestModel, "rf_model")

    print("Logged Random Forest model to MLflow")


2026/07/07 13:45:34 INFO mlflow.tracking.fluent: Experiment with name 'Customer_Spending_Prediction_V1' does not exist. Creating a new experiment.
2026/07/07 13:46:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged Random Forest model to MLflow
🏃 View run RandomForest_Model at: http://mlflow:5000/#/experiments/7/runs/8a8869bb6a734c7591681977a70bf932
🧪 View experiment at: http://mlflow:5000/#/experiments/7


## XGB

In [10]:
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# --------------------------
# XGBoost Spark phân tán
# --------------------------
xgb = SparkXGBRegressor(
    features_col="scaledFeatures",
    label_col=target,
    num_workers=1,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    objective="reg:squarederror"
)

# --------------------------
# Pipeline
# --------------------------
xgb_pipeline = Pipeline(stages=[assembler, scaler, xgb])

# --------------------------
# Grid Search / TrainValidationSplit
# --------------------------
param_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [4, 6])
    .addGrid(xgb.n_estimators, [50, 100])
    .addGrid(xgb.learning_rate, [0.05, 0.1])
    .addGrid(xgb.subsample, [0.7, 0.8])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="r2"
)

tvs = TrainValidationSplit(
    estimator=xgb_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=2,   # chạy song song 4 mô hình
    seed=42
)

# --------------------------
# Huấn luyện mô hình
# --------------------------
xgb_tvs_model = tvs.fit(train_df)

# --------------------------

2026-07-07 13:49:33,514 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 1 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.8, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-07-07 13:49:33,517 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 1 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.7, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-07-07 13:49:39,454 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-07-07 13:49:39,458 INFO XGBoost-PySpark: _fit Finished xgboost training!
2026-07-07 13:49:47,595 INFO XGBoost-PySpark: 

In [11]:
# Lấy mô hình tốt nhất
best_xgb = xgb_tvs_model.bestModel.stages[-1]
tuned_params = ["max_depth", "n_estimators", "learning_rate", "subsample"]
print("Best XGBoost Parameters (from grid search):")
for p in tuned_params:
    value = best_xgb.getOrDefault(best_xgb.getParam(p))
    print(f" - {p}: {value}")

Best XGBoost Parameters (from grid search):
 - max_depth: 4
 - n_estimators: 100
 - learning_rate: 0.1
 - subsample: 0.7


In [12]:
# Đánh giá mô hình
# --------------------------
train_pred = xgb_tvs_model.transform(train_df)
test_pred  = xgb_tvs_model.transform(test_df)

metrics = ['r2', 'mae', 'rmse']
for metric in metrics:
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    print(f"{metric.upper()} Train:", round(evaluator.evaluate(train_pred), 4))
    print(f"{metric.upper()} Test :", round(evaluator.evaluate(test_pred), 4))
    print("----------")

R2 Train: 0.9987
R2 Test : 0.9958
----------
MAE Train: 34.8634
MAE Test : 44.513
----------
RMSE Train: 60.9379
RMSE Test : 102.4116
----------


In [14]:
import mlflow
import mlflow.spark

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("XGBoost_Regression_V1")

with mlflow.start_run():
    mlflow.spark.log_model(best_xgb, "xgb_model_V1")
    mlflow.log_metric("r2_train", 0.9987)
    mlflow.log_metric("r2_test", 0.9905)

2026/07/07 13:54:21 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Regression_V1' does not exist. Creating a new experiment.
2026/07/07 13:54:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run sneaky-dove-418 at: http://mlflow:5000/#/experiments/8/runs/e26e6621419b47ae8bac92659e5d100c
🧪 View experiment at: http://mlflow:5000/#/experiments/8


In [15]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Spending_Prediction_V1")

# Tính metrics train/test
metrics_dict = {}
for metric in metrics:  # metrics = ['r2', 'mae', 'rmse']
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_dict[f"{metric}_train"] = round(evaluator.evaluate(xgb_tvs_model.transform(train_df)), 4)
    metrics_dict[f"{metric}_test"]  = round(evaluator.evaluate(xgb_tvs_model.transform(test_df)), 4)

# Lưu vào MLflow
with mlflow.start_run(run_name="XGBoost_Model_V1"):
    # Log param
    mlflow.log_param("model", "SparkXGBRegressor")
    mlflow.log_param("max_depth", best_xgb.getOrDefault(best_xgb.getParam("max_depth")))
    mlflow.log_param("n_estimators", best_xgb.getOrDefault(best_xgb.getParam("n_estimators")))
    mlflow.log_param("learning_rate", best_xgb.getOrDefault(best_xgb.getParam("learning_rate")))
    mlflow.log_param("subsample", best_xgb.getOrDefault(best_xgb.getParam("subsample")))
    
    # Log metric
    for k, v in metrics_dict.items():
        mlflow.log_metric(k, v)
        print(f"{k}: {v}")
    
    # Log mô hình (PipelineModel đầy đủ: assembler + scaler + XGB)
    mlflow.spark.log_model(xgb_tvs_model.bestModel, "xgb_model_V1")

    print("Logged XGBoost Spark model to MLflow")


r2_train: 0.9987
r2_test: 0.9958
mae_train: 34.8634
mae_test: 44.513
rmse_train: 60.9379
rmse_test: 102.4116


2026/07/07 13:55:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged XGBoost Spark model to MLflow
🏃 View run XGBoost_Model_V1 at: http://mlflow:5000/#/experiments/7/runs/1c5f533dc9914ddb8ea1faf13e0442a0
🧪 View experiment at: http://mlflow:5000/#/experiments/7


In [16]:
mlflow.spark.log_model(
    best_xgb,
    artifact_path="xgb_model_V1",
    registered_model_name="XGB_Model_V1"
)


2026/07/07 13:55:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'XGB_Model_V1'.
2026/07/07 13:55:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_Model_V1, version 1
Created version '1' of model 'XGB_Model_V1'.


In [ ]:
xong